# 04-7. 대용량 스트리밍과 오류 복구

## Goal

작은 교육용 CSV를 한 레코드씩 읽고, 정상 레코드와 복구 가능한 오류를 분리합니다. 입력 스트리밍과 전체 결과의 메모리 사용을 구분하고 다음 계약을 검증합니다.

- `csv.DictReader`와 제너레이터로 레코드를 지연 생성합니다.
- 파일 전체 오류와 한 레코드의 값 오류를 다른 경계에서 처리합니다.
- 오류의 물리 행 끝 위치, 파싱된 값, 예외 유형과 메시지를 보존합니다.
- 정상·오류 건수의 보존 법칙과 JSON Lines 순차 출력을 확인합니다.

## Setup

모든 입력 파일은 `TemporaryDirectory` 안에서 생성합니다. 실제 업무 파일이나 저장소 파일을 읽거나 변경하지 않으며, 마지막 점검에서 임시 작업 공간을 정리합니다. Python 3.10 이상과 표준 라이브러리만 사용합니다.

In [ ]:
from __future__ import annotations

import csv
import json
import sys
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory

assert sys.version_info >= (3, 10), "Python 3.10 이상이 필요합니다"

_lab = TemporaryDirectory(prefix="chapter-04-7-")
LAB_DIR = Path(_lab.name).resolve()
ITEMS_PATH = LAB_DIR / "items.csv"

print("임시 실습 디렉터리:", LAB_DIR)

## Steps

### 1. 재현 가능한 CSV fixture 만들기

정상 레코드 2건과 숫자 변환·범위 오류 레코드 2건을 만듭니다. 파일 위치를 임시 실습 디렉터리 안으로 고정해 반복 실행의 영향을 제한합니다.

In [ ]:
ITEMS_PATH.write_text(
    "name,price,quantity\n"
    "키보드,30000,2\n"
    "마우스,abc,1\n"
    "모니터,200000,-1\n"
    "USB,10000,3\n",
    encoding="utf-8",
)

assert ITEMS_PATH.is_file()
assert ITEMS_PATH.is_relative_to(LAB_DIR)
print(ITEMS_PATH.read_text(encoding="utf-8"))

### 2. 파일 계약과 레코드 계약 분리하기

`csv_rows()`는 헤더와 CSV 구조처럼 파일 전체에 적용되는 계약을 검사한 뒤 `(물리 행 끝 번호, 레코드)`를 하나씩 생성합니다. `parse_item()`은 레코드 하나의 필수값·자료형·범위만 검증합니다.

In [ ]:
REQUIRED_FIELDS = {"name", "price", "quantity"}


def parse_item(row: dict) -> dict:
    if None in row:
        raise ValueError("헤더보다 값이 많은 레코드입니다")
    if any(value is None for value in row.values()):
        raise ValueError("필드가 누락된 레코드입니다")

    name = row["name"].strip()
    price_text = row["price"]
    quantity_text = row["quantity"]

    if not name:
        raise ValueError("상품명이 비어 있습니다")
    price = int(price_text)
    quantity = int(quantity_text)
    if price < 0 or quantity < 0:
        raise ValueError("가격과 수량은 0 이상이어야 합니다")

    return {
        "name": name,
        "price": price,
        "quantity": quantity,
        "total": price * quantity,
    }


def csv_rows(path: Path):
    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file, strict=True)
        fieldnames = reader.fieldnames

        if fieldnames is None:
            raise ValueError("CSV 헤더가 없습니다")

        missing = REQUIRED_FIELDS - set(fieldnames)
        if missing:
            names = ", ".join(sorted(missing))
            raise ValueError(f"필수 헤더가 없습니다: {names}")
        if len(fieldnames) != len(set(fieldnames)):
            raise ValueError("중복된 CSV 헤더가 있습니다")

        for row in reader:
            yield reader.line_num, row


SENSITIVE_FIELDS = {"password", "token", "secret", "api_key"}


def redact_row(row: dict) -> dict:
    return {
        key: "<redacted>" if key.casefold() in SENSITIVE_FIELDS else value
        for key, value in row.items()
    }


def process_rows(path: Path) -> tuple[list[dict], list[dict]]:
    records = []
    errors = []

    for line_end, row in csv_rows(path):
        try:
            records.append(parse_item(row))
        except (KeyError, TypeError, ValueError) as exc:
            errors.append({
                "line_end": line_end,
                "row": redact_row(row),
                "error_type": type(exc).__name__,
                "message": str(exc),
            })

    return records, errors

### 3. 오류를 보존하며 전체 입력 처리하기

레코드 오류가 나도 다음 레코드를 계속 처리합니다. 재현에 필요한 필드는 보존하되 비밀번호·토큰 같은 민감 필드는 오류 보고서에서 마스킹합니다. 다만 이 기본 구현은 입력만 순차적으로 읽을 뿐 `records`와 `errors`를 리스트에 누적하므로 전체 과정이 고정 메모리인 것은 아닙니다.

In [ ]:
records, errors = process_rows(ITEMS_PATH)
summary = {
    "processed": len(records) + len(errors),
    "valid": len(records),
    "errors": len(errors),
    "total_amount": sum(record["total"] for record in records),
}

print(json.dumps({"summary": summary, "records": records}, ensure_ascii=False, indent=2))
print(json.dumps({"errors": errors}, ensure_ascii=False, indent=2))

### 4. 논리 레코드와 물리 행 번호 구분하기

따옴표 안의 줄바꿈은 한 CSV 레코드에 포함될 수 있습니다. `reader.line_num`은 그 레코드를 읽은 뒤의 마지막 물리 행 번호를 제공합니다.

In [ ]:
MULTILINE_PATH = LAB_DIR / "multiline.csv"
MULTILINE_PATH.write_text(
    'name,price,quantity\n"키\n보드",30000,2\n',
    encoding="utf-8",
)

multiline_rows = list(csv_rows(MULTILINE_PATH))
multiline_line_ends = [line_end for line_end, _ in multiline_rows]
print("레코드 마지막 물리 행:", multiline_line_ends)
print("파싱된 상품명:", multiline_rows[0][1]["name"])

### 5. JSON Lines를 한 건씩 직렬화하기

여기서는 실제 결과 파일 대신 메모리 버퍼로 순차 출력 형식만 검증합니다. 최종 파일에 저장할 때는 04-8의 임시 파일 교체 절차가 필요합니다.

In [ ]:
def write_json_lines(events, file) -> None:
    for kind, payload in events:
        json.dump(
            {"kind": kind, "data": payload},
            file,
            ensure_ascii=False,
            allow_nan=False,
        )
        file.write("\n")


events = (
    [("record", record) for record in records]
    + [("error", error) for error in errors]
)
buffer = StringIO()
write_json_lines(events, buffer)
jsonl_values = [json.loads(line) for line in buffer.getvalue().splitlines()]
print(json.dumps(jsonl_values, ensure_ascii=False, indent=2))

## Checks

정상·오류 건수와 합계, 여러 줄 필드의 위치, JSON Lines 보존 법칙을 확인합니다. 필수 헤더 누락과 복구할 수 없는 CSV 구조 오류는 레코드 오류 목록에 넣지 않고 호출자에게 전달되는지도 검증합니다.

In [ ]:
assert summary == {
    "processed": 4,
    "valid": 2,
    "errors": 2,
    "total_amount": 90_000,
}
assert summary["processed"] == summary["valid"] + summary["errors"]
assert {error["line_end"] for error in errors} == {3, 4}
assert redact_row({"username": "alice", "password": "guess"}) == {
    "username": "alice",
    "password": "<redacted>",
}
assert multiline_line_ends == [3]
assert len(jsonl_values) == summary["processed"]
assert [value["kind"] for value in jsonl_values].count("record") == 2
assert [value["kind"] for value in jsonl_values].count("error") == 2

MISSING_HEADER_PATH = LAB_DIR / "missing-header.csv"
MISSING_HEADER_PATH.write_text("name,price\n키보드,30000\n", encoding="utf-8")
try:
    list(csv_rows(MISSING_HEADER_PATH))
except ValueError as exc:
    missing_header_message = str(exc)
else:
    raise AssertionError("필수 헤더 누락을 거부해야 합니다")
assert "quantity" in missing_header_message

BROKEN_CSV_PATH = LAB_DIR / "broken.csv"
BROKEN_CSV_PATH.write_text(
    'name,price,quantity\n키보드,30000,"닫히지 않은 값\n',
    encoding="utf-8",
)
try:
    process_rows(BROKEN_CSV_PATH)
except csv.Error as exc:
    csv_structure_error = str(exc)
else:
    raise AssertionError("복구할 수 없는 CSV 구조 오류가 발생해야 합니다")
assert csv_structure_error

print("04-7 계약 검증 통과")

In [ ]:
temporary_root = LAB_DIR
_lab.cleanup()
assert not temporary_root.exists()
print("임시 실습 디렉터리 정리 완료")

## Next Steps

- 레코드 수가 매우 크다면 정상·오류 목록을 메모리에 누적하지 않고 각각 JSON Lines로 순차 저장합니다.
- 비밀번호·토큰·개인정보가 있는 필드는 오류 보고서에 그대로 복사하지 않는 정책을 추가합니다.
- 다음 절에서는 JSON Lines와 보고서를 고유한 임시 파일에 완성한 뒤 최종 경로로 교체합니다.